<div style="width: 100%; overflow: hidden;">
    <div style="width: 150px; float: left;"> <img src="data/D4Sci_logo_ball.png" alt="Data For Science, Inc" align="left" border="0"> </div>
    <div style="float: left; margin-left: 10px;"> <h1>Gemini API and VertexAI</h1>
<h1>Fundamentals</h1>
        <p>Bruno Gonçalves<br/>
        <a href="http://www.data4sci.com/">www.data4sci.com</a><br/>
            @bgoncalves, @data4sci</p></div>
</div>

### Notebook Summary

**What this notebook teaches:**
This notebook introduces the fundamentals of the Gemini model family. It covers the basics of using the Gemini API, listing available models, generating text content, checking safety ratings, and the differences between the Google GenAI SDK and the Vertex AI SDK.

**GCP/Gemini APIs used:**
- `google.genai` SDK
- `vertexai` SDK
- Gemini models (e.g., `gemini-2.5-flash`)

**Prerequisites:**
- Google Cloud Project with Vertex AI API enabled.
- API Key for `google.genai` or ADC (Application Default Credentials) for `vertexai`.
- Installed libraries: `google-genai`, `vertexai`, `pandas`, `matplotlib`.

**Expected Output:**
A thorough understanding of how to connect to Gemini models using both direct API and Vertex AI, list available models, perform basic content generation, and understand the core differences between the SDKs.


In [1]:
from collections import Counter
from pprint import pprint
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import google.genai as genai
from google.genai import types
import vertexai
from vertexai.generative_models import GenerativeModel, Part, GenerationConfig

import watermark
%load_ext watermark
%matplotlib inline


We start by printing out the versions of the libraries we're using for future reference

In [2]:
%watermark -n -v -m -g -iv

Python implementation: CPython
Python version       : 3.12.10
IPython version      : 9.10.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 25.3.0
Machine     : arm64
Processor   : arm
CPU cores   : 16
Architecture: 64bit

Git hash: 96f1c1d80a1dfb48c21e951377f73395e1195d85

matplotlib: 3.10.8
numpy     : 2.4.2
pandas    : 3.0.1
vertexai  : 1.71.1
watermark : 2.6.0



Load default figure style

In [3]:
plt.style.use('d4sci.mplstyle')
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

# 1. Introduction to the Gemini API

## The Gemini Model Family

Google's **Gemini** is a family of natively multimodal foundation models capable of processing text, images, audio, video, and code.

NOTE: Set your API key before running this cell:

   export GEMINI_API_KEY='your-key-here'
 
 Or set it programmatically (not recommended for shared notebooks):
   
   os.environ["GEMINI_API_KEY"] = "your-key-here"

In [4]:
api_key = os.environ.get("GEMINI_API_KEY")

if not api_key:
    raise EnvironmentError(
        "GEMINI_API_KEY environment variable not set. "
        "Please set it before running this notebook."
    )

client = genai.Client(api_key=api_key)
print("Gemini API configured successfully.")


Gemini API configured successfully.


In [5]:
# List all available Gemini models
print("Available models:")
print("-" * 60)

for model in client.models.list():
    if "generateContent" in (model.supported_actions or []):
        print(f"  Name       : {model.name}")
        print(f"  Display    : {model.display_name}")
        print(f"  Description: {model.description[:80]}..." if model.description and len(model.description) > 80 else f"  Description: {model.description}")
        print()


Available models:
------------------------------------------------------------
  Name       : models/gemini-2.5-flash
  Display    : Gemini 2.5 Flash
  Description: Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports ...

  Name       : models/gemini-2.5-pro
  Display    : Gemini 2.5 Pro
  Description: Stable release (June 17th, 2025) of Gemini 2.5 Pro

  Name       : models/gemini-2.0-flash
  Display    : Gemini 2.0 Flash
  Description: Gemini 2.0 Flash

  Name       : models/gemini-2.0-flash-001
  Display    : Gemini 2.0 Flash 001
  Description: Stable version of Gemini 2.0 Flash, our fast and versatile multimodal model for ...

  Name       : models/gemini-2.0-flash-lite-001
  Display    : Gemini 2.0 Flash-Lite 001
  Description: Stable version of Gemini 2.0 Flash-Lite

  Name       : models/gemini-2.0-flash-lite
  Display    : Gemini 2.0 Flash-Lite
  Description: Gemini 2.0 Flash-Lite

  Name       : models/gemini-2.5-flash-preview-tts
  Display    : Gem

Inspect a specific model's properties

In [6]:
model_info = client.models.get(model="models/gemini-2.5-flash")

print(f"Model Name          : {model_info.name}")
print(f"Display Name        : {model_info.display_name}")
print(f"Input Token Limit   : {model_info.input_token_limit:,}")
print(f"Output Token Limit  : {model_info.output_token_limit:,}")
print(f"Supported Actions   : {model_info.supported_actions}")
print(f"Temperature Range   : {model_info.temperature} (default)")
print(f"TopP                : {model_info.top_p}")
print(f"TopK                : {model_info.top_k}")


Model Name          : models/gemini-2.5-flash
Display Name        : Gemini 2.5 Flash
Input Token Limit   : 1,048,576
Output Token Limit  : 65,536
Supported Actions   : ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
Temperature Range   : 1.0 (default)
TopP                : 0.95
TopK                : 64


## First API Call: Simple Text Generation

Create a model instance and generate a simple response

In [7]:
prompt = "Explain the concept of attention mechanisms in neural networks in 3 sentences."

response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)

print("Prompt:")
print(f"  {prompt}")
print()
print("Response:")
print(response.text)


Prompt:
  Explain the concept of attention mechanisms in neural networks in 3 sentences.

Response:
Attention mechanisms enable neural networks to dynamically focus on the most relevant parts of their input data when processing information, rather than treating all parts equally. This is achieved by learning a set of weights that determine the "importance" or "relevance" of different input elements for generating a specific output. By selectively emphasizing crucial information and suppressing irrelevant details, attention significantly improves model performance, particularly in tasks involving long sequences like machine translation or image analysis.


## Generation Configuration

Using `GenerationConfig` to control sampling behavior

In [8]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=(
        "List the top 5 use cases for large language models in enterprise settings. "
        "Format as a numbered list with a one-sentence explanation for each."
    ),
    config=types.GenerateContentConfig(
        temperature=0.2,          # Low temperature for factual, consistent output
        top_p=0.9,                # Nucleus sampling
        top_k=40,                 # Restrict to top 40 tokens
        max_output_tokens=512,    # Limit response length
        stop_sequences=["##"],    # Stop if the model produces '##'
    )
)

print(response.text)
print()
print("--- Usage Metadata ---")
print(f"Prompt tokens    : {response.usage_metadata.prompt_token_count}")
print(f"Candidates tokens: {response.usage_metadata.candidates_token_count}")
print(f"Total tokens     : {response.usage_metadata.total_token_count}")


Here are the top 5 use cases for large language models in enterprise settings:

1.  

--- Usage Metadata ---
Prompt tokens    : 30
Candidates tokens: 19
Total tokens     : 537


## Safety Settings

Gemini includes built-in safety filters that can be adjusted per harm category:

| Category | Description |
|---|---|
| `HARM_CATEGORY_HARASSMENT` | Bullying, threatening, or abusive content |
| `HARM_CATEGORY_HATE_SPEECH` | Content targeting identity characteristics |
| `HARM_CATEGORY_SEXUALLY_EXPLICIT` | Explicit sexual content |
| `HARM_CATEGORY_DANGEROUS_CONTENT` | Content promoting dangerous activities |

**Block thresholds** (from most to least restrictive):
`BLOCK_LOW_AND_ABOVE` > `BLOCK_MEDIUM_AND_ABOVE` > `BLOCK_ONLY_HIGH` > `BLOCK_NONE`

In [9]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="What are the main risks of deploying AI in healthcare?",
    config=types.GenerateContentConfig(
        safety_settings=[
            types.SafetySetting(category="HARM_CATEGORY_HARASSMENT",        threshold="BLOCK_MEDIUM_AND_ABOVE"),
            types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH",       threshold="BLOCK_MEDIUM_AND_ABOVE"),
            types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="BLOCK_MEDIUM_AND_ABOVE"),
            types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="BLOCK_MEDIUM_AND_ABOVE"),
        ]
    )
)

# Check safety ratings (may be None on some models/API versions)
print("Safety Ratings:")
if not response.candidates:
    print("  (no candidates — e.g. blocked or empty response)")
else:
    ratings = response.candidates[0].safety_ratings
    if ratings:
        for rating in ratings:
            print(f"  {rating.category.name}: {rating.probability.name}")
    else:
        print("  (not returned for this response)")
print()
print("Response:")
print(response.text)


Safety Ratings:
  (not returned for this response)

Response:
Deploying Artificial Intelligence (AI) in healthcare holds immense promise for improving diagnosis, treatment, research, and administrative efficiency. However, it also introduces several significant risks that need careful consideration and mitigation strategies. Here are the main risks:

1.  **Patient Safety and Clinical Accuracy:**
    *   **Misdiagnosis/Incorrect Treatment:** AI models, especially if poorly trained or using flawed data, can provide incorrect diagnoses or treatment recommendations, leading to patient harm or suboptimal care.
    *   **Alert Fatigue:** Over-reliance on AI systems that generate too many alerts (false positives) can lead clinicians to ignore critical warnings.
    *   **Software Malfunctions:** Bugs or errors in AI algorithms or the systems they integrate with can directly lead to patient harm.
    *   **Lack of Generalizability:** An AI model trained on data from one specific population, ho

## Multi-turn Conversations (Chat)

The Gemini SDK supports stateful multi-turn conversations via `start_chat()`.

In [10]:
chat = client.chats.create(model="gemini-2.5-flash")

turns = [
    "I'm learning about neural networks. Can you briefly explain what a neuron does?",
    "How does that relate to the transformer architecture?",
    "And what makes Gemini different from earlier transformers?",
]

for user_message in turns:
    print(f"USER: {user_message}")
    response = chat.send_message(user_message)
    print(f"GEMINI: {response.text}")
    print("-" * 60)

print(f"\nTotal turns in history: {len(chat.get_history())}")


USER: I'm learning about neural networks. Can you briefly explain what a neuron does?
GEMINI: A neuron in a neural network is like a tiny decision-making unit, inspired by biological neurons in the brain. Here's what it does, step-by-step:

1.  **Receives Inputs:** It takes in one or more numerical values from other neurons or directly from the input data.
2.  **Applies Weights:** Each input is multiplied by a corresponding **weight**. Weights represent the "importance" or "strength" of that particular input. A higher weight means that input has a greater influence on the neuron's output.
3.  **Adds Bias:** An additional value called **bias** is added to the sum of the weighted inputs. The bias allows the neuron to activate even if all its inputs are zero, or to shift its activation threshold.
4.  **Sums Everything Up:** All the weighted inputs are summed together, and then the bias is added to this sum. This is often called the "net input" or "pre-activation."
5.  **Passes Through an 

# 3. Overview of Vertex AI

**Vertex AI** is Google Cloud's unified machine learning platform. It provides end-to-end ML capabilities from data preparation and model training to deployment, monitoring, and MLOps — all under one roof.

**Prerequisites:**
1. Google Cloud project with billing enabled
2. Enable the Vertex AI API: `gcloud services enable aiplatform.googleapis.com`
3. Authenticate: `gcloud auth application-default login`
4. Install the SDK: `pip install google-cloud-aiplatform`
5. Set environment variables:
   ```bash
   export GCP_PROJECT='your-project-id'
   export GCP_LOCATION='us-central1'
   ```

In [11]:
GCP_PROJECT  = os.environ.get("GCP_PROJECT")
GCP_LOCATION = os.environ.get("GCP_LOCATION", "us-central1")

if not GCP_PROJECT:
    raise EnvironmentError(
        "GCP_PROJECT environment variable not set. "
        "Please set it to your Google Cloud project ID."
    )

# Initialize the Vertex AI SDK
vertexai.init(project=GCP_PROJECT, location=GCP_LOCATION)

print(f"Vertex AI initialized.")
print(f"  Project  : {GCP_PROJECT}")
print(f"  Location : {GCP_LOCATION}")

Vertex AI initialized.
  Project  : gen-lang-client-0667691803
  Location : us-central1


In [12]:
# Using the Vertex AI GenerativeModel
vertex_model = GenerativeModel("gemini-2.5-flash")

vertex_response = vertex_model.generate_content(
    "What are the three most important considerations when deploying LLMs in production?"
)
print("Response via Vertex AI:")
print(vertex_response.text)

Response via Vertex AI:
Deploying Large Language Models (LLMs) in production involves unique challenges beyond traditional software or even ML model deployment. Here are the three most important considerations:

1.  **Responsible AI & Output Quality (Safety, Fairness, Hallucinations, Bias):**
    This is paramount because LLMs generate content, and their outputs can have significant real-world impact.
    *   **Hallucinations & Accuracy:** LLMs can confidently generate incorrect, nonsensical, or made-up information. Ensuring factual accuracy (e.g., through Retrieval Augmented Generation - RAG) and managing the risk of hallucinations is critical for trust and reliability.
    *   **Bias & Fairness:** LLMs are trained on vast datasets that reflect societal biases. These biases can be amplified and manifest in discriminatory or unfair outputs. Careful evaluation, mitigation strategies, and monitoring are essential.
    *   **Safety & Harmful Content:** LLMs can be prompted to generate hat

In [13]:
# Generation config with Vertex AI
vertex_config = GenerationConfig(
    temperature=0.3,
    top_p=0.95,
    top_k=40,
    max_output_tokens=1024,
)

vertex_model_configured = GenerativeModel(
    "gemini-2.5-flash",
    generation_config=vertex_config,
)

structured_prompt = """You are an expert ML engineer. 
Provide a concise checklist (5 items) for productionizing a generative AI application.
Format: numbered list, one line per item."""

vertex_response = vertex_model_configured.generate_content(structured_prompt)

print("Structured response via Vertex AI (with config):")
print(vertex_response.text)
print()
print("--- Usage Metadata ---")
print(vertex_response.usage_metadata)

Structured response via Vertex AI (with config):
Here is a concise checklist for productionizing a generative AI application:

1.  **Robust Deployment & Scalability:** Ensure reliable, low-latency model serving with auto-scaling to handle varying loads.
2.  **Comprehensive Safety & Guardrails:** Implement content moderation, prompt injection defenses, and ethical output filters.
3.  **Real-time Monitoring & Observability:** Track model performance, output quality, cost, and user engagement with alerts.
4.  **Continuous Evaluation & Feedback Loops:** Establish human feedback, A/B testing, and iterative model improvement mechanisms.
5.  **Cost Optimization & Resource Management:** Optimize inference costs, manage GPU resources, and select appropriate model sizes.

--- Usage Metadata ---
prompt_token_count: 36
candidates_token_count: 144
total_token_count: 917



In [14]:
# System instructions: Vertex AI supports system-level persona
system_instruction_model = GenerativeModel(
    "gemini-2.5-flash",
    system_instruction="""You are a senior data scientist at a Fortune 500 company.
    You communicate in a concise, technical, and results-focused manner.
    Always back up your statements with data or industry best practices."""
)

response = system_instruction_model.generate_content(
    "What is the most effective way to evaluate an LLM for a customer support use case?"
)

print("Response with system instruction:")
print(response.text)

Response with system instruction:
To effectively evaluate an LLM for a customer support use case, a multi-faceted, hybrid approach combining robust offline assessment with rigorous online A/B testing is essential. This ensures both intrinsic quality and measurable business impact.

### 1. Define Success Metrics

Prioritize both business and technical metrics upfront:

*   **Business Metrics:** Customer Satisfaction (CSAT), First Contact Resolution (FCR), Mean Time To Resolution (MTTR), Agent Escalation Rate, Deflection Rate.
*   **Technical Metrics:** Accuracy, Hallucination Rate, Latency, Adherence to Brand Guidelines.

### 2. Offline Evaluation (Pre-deployment Risk Mitigation)

This phase focuses on intrinsic quality and readiness before exposure to live customers.

*   **Curated Test Datasets:**
    *   **Source:** Develop a representative dataset of historical customer queries, agent responses, and internal knowledge base articles. Ensure diversity across common topics, edge cases,

## Comparing API Calls: Direct API vs Vertex AI

Both paths use nearly identical Python interfaces — the main difference is initialization and authentication:

**Direct API (google-genai):**
```python
import google.genai as genai

client = genai.Client(api_key="YOUR_API_KEY")
response = client.models.generate_content(model="gemini-2.0-flash", contents="Hello, Gemini!")
print(response.text)
```

**Vertex AI (google-cloud-aiplatform):**
```python
import vertexai
from vertexai.generative_models import GenerativeModel

vertexai.init(project="your-project", location="us-central1")
model = GenerativeModel("gemini-2.0-flash")
response = model.generate_content("Hello, Gemini!")
print(response.text)
```

The core API surface (`generate_content`, `GenerationConfig`, `start_chat`) is essentially the same in both SDKs, making migration between the two straightforward.


In [15]:
# Side-by-side token counting: both APIs expose count_tokens

test_prompt = "Explain what Vertex AI is in one paragraph."

# Direct API
direct_count = client.models.count_tokens(model="gemini-2.5-flash", contents=test_prompt)
print(f"Direct API - Token count for test prompt: {direct_count.total_tokens}")

# Vertex AI
vertex_count = vertex_model.count_tokens(test_prompt)
print(f"Vertex AI  - Token count for test prompt: {vertex_count.total_tokens}")


Direct API - Token count for test prompt: 10
Vertex AI  - Token count for test prompt: 9


<center>
     <img src="data/D4Sci_logo_full.png" alt="Data For Science, Inc" align="center" border="0" width=300px> 
</center>